# FastSurfer seg_only on IXI (Kaggle, resumable across runs)

Same preprocessing as `notebooks/fastsurfer_colab.ipynb` (sections 1-6), for Kaggle: all 563 subjects in one session, no Google Drive.

**Settings (right panel):** Accelerator **GPU T4 x2** (not P100) · Internet **on** (needs a phone-verified account).

**Subjects already done are never recomputed.** Every input of this notebook is searched for finished subjects (`caps_fastsurfer_2mm/subjects/.../*_desc-{nu,brain}_res-2x2x2_T1w.nii.gz`); they are copied to the output folder first, and the script skips every subject whose two images exist.
- **First run:** add as input a Kaggle dataset made from the Drive folder `MyDrive/test-clinicadl/caps_fastsurfer_2mm` (download it as a zip from Drive, then Kaggle → Datasets → New dataset → upload the zip).
- **Next runs:** also add the output of the previous version of this notebook (Add Input → Your Work → this notebook).

**Run in the background:** Save Version → *Save & Run All (Commit)*. Kaggle stops a session after 12 h: the preprocessing stops by itself after 11 h, so the outputs of every finished subject are saved (Output tab, `caps_fastsurfer_2mm/`). Run new versions until cell 5 shows 563 / 563.

Large temporary files (IXI, FastSurfer) are kept in `/tmp`, so only the results are saved as output (~1.5 GB at the end).

## 1. Paths, scripts, and subjects already done

In [ ]:
import os, glob, shutil, filecmp
import pandas as pd
os.environ['MPLBACKEND'] = 'Agg'

WORK = '/tmp/test-clinicadl'                         # scripts + IXI (temporary, not saved)
FS_OUT = '/kaggle/working/caps_fastsurfer_2mm'       # results (saved as the notebook output)
REPO = 'https://github.com/Ayoub-TAREK/cross-site-brain-age.git'
if not os.path.exists(WORK):
    !git clone -q {REPO} {WORK}
else:
    !git -C {WORK} pull -q
!git -C {WORK} log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

# copy the finished subjects (and logs / stats) found in the inputs; nothing already in FS_OUT is overwritten
os.makedirs(FS_OUT, exist_ok=True)
brains = glob.glob('/kaggle/input/**/subjects/*/*/fastsurfer/*_desc-brain_res-2x2x2_T1w.nii.gz', recursive=True)
roots = sorted({os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(f))))) for f in brains})
for k, root in enumerate(roots):
    copied = 0
    for src in glob.glob(f'{root}/**/*', recursive=True):
        if os.path.isdir(src) or os.path.basename(src).startswith('tmp_'):
            continue
        dst = os.path.join(FS_OUT, os.path.relpath(src, root))
        if os.path.exists(dst):
            if src.endswith('.tsv') and not filecmp.cmp(src, dst, shallow=False):   # same log name, other content
                dst = dst[:-4] + f'_in{k}.tsv'
            else:
                continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        copied += 1
    print(f'input {root}: {copied} files copied')
done = len(glob.glob(f'{FS_OUT}/subjects/*/*/fastsurfer/*_desc-brain_res-2x2x2_T1w.nii.gz'))
print(f'\n{done} / 563 subjects already done: they will be skipped')

## 2. Download IXI and convert to BIDS (≈ 4.5 GB, a few minutes)

In [ ]:
!pip install -q xlrd
os.makedirs(f'{WORK}/data/IXI/raw', exist_ok=True)
%cd {WORK}/data/IXI
URL = 'http://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI'
if not os.path.exists('IXI.xls'):
    !curl -fsSL --retry 3 -o IXI.xls {URL}/IXI.xls
if len(glob.glob('raw/*.nii.gz')) < 500:
    !curl -fsSL --retry 3 -o IXI-T1.tar {URL}/IXI-T1.tar
    !tar -xf IXI-T1.tar -C raw && rm IXI-T1.tar
%cd {WORK}
!python scripts/ixi_to_bids.py

## 3. Install FastSurfer (≈ 5 min)

Own environment (`/tmp/FastSurfer/.venv`, Python 3.12, PyTorch with CUDA). Must end with `CUDA True` and no error.

In [ ]:
FS_HOME = '/tmp/FastSurfer'
FS_PY = f'{FS_HOME}/.venv/bin/python'
if not os.path.exists(f'{FS_HOME}/.installed'):
    !rm -rf {FS_HOME}
    !pip install -q uv
    !git -c advice.detachedHead=false clone -q --depth 1 --branch v2.6.1 https://github.com/Deep-MI/FastSurfer.git {FS_HOME}
    !cd {FS_HOME} && uv venv -q --python 3.12
    req = 'requirements.txt' if os.path.exists(f'{FS_HOME}/requirements.txt') else 'pyproject.toml'
    print('Installing the dependencies from', req)
    !cd {FS_HOME} && uv pip install --python {FS_PY} --torch-backend auto -r {req} 2>&1 | tail -n 5
    !cd {FS_HOME} && PYTHONPATH={FS_HOME} {FS_PY} FastSurferCNN/download_checkpoints.py --all 2>&1 | tail -n 3
    !{FS_PY} -c "import torch, requests, nibabel; assert torch.cuda.is_available()" && touch {FS_HOME}/.installed
!{FS_PY} -c "import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())"
assert os.path.exists(f'{FS_HOME}/.installed'), 'FastSurfer install failed: check the output above (GPU T4 enabled?)'

## 4. All remaining subjects (stops by itself after 11 h)

Subjects already done are skipped. One line per subject: time, brain mask volume, hours left.

In [ ]:
%cd {WORK}
!python scripts/fastsurfer_preprocess.py --fastsurfer-home {FS_HOME} --out-dir {FS_OUT} --max-hours 11

## 5. Check the outputs

In [ ]:
n = len(glob.glob(f'{FS_OUT}/subjects/*/*/fastsurfer/*_desc-brain_res-2x2x2_T1w.nii.gz'))
print(n, 'subjects done (expected 563)' + ('' if n == 563 else ': run a new version with this output as input'))
log = pd.concat([pd.read_csv(f, sep='\t') for f in glob.glob(f'{FS_OUT}/fastsurfer_log*.tsv')])
log = log.drop_duplicates(['participant_id', 'session_id'], keep='last')
log = log.merge(pd.read_csv(f'{FS_OUT}/participants_sessions_age.tsv', sep='\t')[['participant_id', 'site']])
print(log.status.value_counts().to_string())
ok = log[log.status == 'ok']
print(ok.groupby('site').agg(n=('seconds', 'size'), minutes=('seconds', lambda s: round(s.mean() / 60, 1)),
                             mask_ml_mean=('mask_ml', 'mean'), mask_ml_min=('mask_ml', 'min'),
                             mask_ml_max=('mask_ml', 'max')).round(0).to_string())
!du -sh {FS_OUT}